# `ptof_obs_setup_seed`

## What this notebook does
Hand-maintained reference-data seeding and one-off maintenance for the observability pipeline.
It is **not part of the scheduled `obs_fresh_scan` job** — it's run manually, by a human, when a
new capability needs registering, a threshold needs recording, or a one-time cleanup/backfill is
needed. Every other notebook in this repo *reads* the tables this notebook seeds; nothing else
writes to them.

## Prod migration (2026-09-10)
This notebook seeds reference data for the 4 prod capabilities tracked via
`mq_gmdf_dp_prd.oil.ptof_primary__ai_shift_outputs`:
- `saa-display`, `situational-awareness`, `sev2-insights`, `summary`

Old dev capabilities (`saa_insight`, `sev2_insight`, `watchout_narratives`) are deactivated (not
deleted) for historical reference. Detectors dropped in the migration (latency, error rate,
hallucination, transport violations, prompt size, credential fastfail, rapid human correction)
have their output tables dropped in the cleanup cell and their threshold_basis entries marked
`not_applicable_prod`.

## Position in the pipeline
- **Not in any job DAG.** Run interactively/manually, occasionally, not on a schedule.
- **Downstream readers:** `ptof_obs_liveness_detection`, `ptof_obs_mal_output`,
  `ptof_obs_behavioral_correlation` all join against `capability_registry`.
  `ptof_obs_alert.ipynb` reads `threshold_basis` for triage context on Teams cards and
  creates/updates `obs_incidents`.

## Write safety
`capability_registry` and `threshold_basis` are seeded via `CREATE TABLE IF NOT EXISTS` +
a `LEFT ANTI JOIN` insert keyed on each table's natural key — re-running is always a no-op
against rows that already exist.

| Cell | Writes | Idempotent? |
|---|---|---|
| 1 | `capability_registry` | Yes — anti-join insert + deactivate old + update summary |
| 2 | `obs_incidents` schema | Yes — `CREATE TABLE IF NOT EXISTS` |
| 3 | `threshold_basis` | Yes — anti-join insert on check_name |
| 4 | orphaned tables (drop) | Yes — all `DROP TABLE/VIEW IF EXISTS` |

In [ ]:
%sql
-- SAFE to re-run: CREATE TABLE IF NOT EXISTS + anti-join insert, guarded per capability.
-- Seeds capability_registry with all historical rows (dev, now inactive) and the 4 active
-- prod capabilities. The deactivate UPDATE and summary UPDATE at the bottom are idempotent.
--
-- Prod capabilities (active = true):
--   saa-display          — SAA shift display output (~2,633 rows), 2h silence grace
--   situational-awareness — SAA situational awareness (~2,633 rows), 2h silence grace
--   sev2-insights        — SAA sev2 insights (~572 rows), irregular cadence → no silence grace,
--                           silence_ceiling_hours added 2026-09-21 (item #2) as WARN/168h;
--                           promoted CRITICAL and tightened to 120h 2026-09-21 (same day,
--                           FP/FN bias review follow-up, signed off) -- both values have the
--                           same 0/989 empirical historical false-positive rate (max gap ever
--                           observed 102.5h), but 120h cuts the false-negative exposure window
--                           by ~29% vs 168h.
--   summary               — ISH EOS handover summary (~53 rows), 36h silence grace
--
-- is_groundable = false for all prod capabilities: hallucination detection is dropped because
-- the regex+similarity approach flags computed numbers as hallucinated (4.9% false positive rate).
CREATE TABLE IF NOT EXISTS mq_gmdf_dev.oil_obs.capability_registry (
    capability           STRING,
    is_generative        BOOLEAN,
    is_gxp_relevant      BOOLEAN,
    expected_min_daily   INT,
    required_fields      ARRAY<STRING>,
    owner                STRING,
    active               BOOLEAN,
    notes                STRING,
    silence_grace_hours  INT,
    is_groundable        BOOLEAN
);

-- Added 2026-09-21 (item #2, signed off): silence_ceiling_hours backstop column, distinct from
-- silence_grace_hours. NULL for every capability except sev2-insights -- see the
-- capability_silence_ceiling detector in ptof_obs_liveness_detection.ipynb for why this exists
-- (silence_grace_hours IS NULL capabilities are structurally excluded from capability_silence,
-- so without this there was no backstop at any length). Safe to re-run: no-op if it already exists.
ALTER TABLE mq_gmdf_dev.oil_obs.capability_registry
  ADD COLUMN IF NOT EXISTS silence_ceiling_hours INT;

-- Insert all capabilities (both historical dev and new prod) via anti-join so re-running is safe.
INSERT INTO mq_gmdf_dev.oil_obs.capability_registry
  (capability, is_generative, is_gxp_relevant, expected_min_daily, owner, active, notes,
   silence_grace_hours, is_groundable)
SELECT t.* FROM VALUES
  -- historical dev capabilities (inactive, kept for reference)
  ('dsa_batch_summary',  true,  true,  0,  'dsa-team',  false,
     'vertex26 · dev-only, deactivated for prod migration 2026-09-10', NULL, false),
  ('dsa_compare',        true,  false, 0,  'dsa-team',  false,
     'vertex26 · dev-only, deactivated for prod migration 2026-09-10', NULL, false),
  ('dsa_copilot',        true,  false, 20, 'dsa-team',  false,
     'vertex26 · dev-only, deactivated for prod migration 2026-09-10', NULL, false),
  ('dsa_copilot_step',   true,  false, 0,  'unassigned', false,
     'cortex · dev-only, deactivated for prod migration 2026-09-10', NULL, false),
  ('dsa_optimize',       true,  false, 10, 'dsa-team',  false,
     'dev-only, deactivated for prod migration 2026-09-10', NULL, false),
  ('dsa_session_summary',true,  false, 0,  'unassigned', false,
     'cortex · dev-only, deactivated for prod migration 2026-09-10', NULL, false),
  ('probe',              true,  false, 0,  'unassigned', false,
     'cortex · dev-only, deactivated for prod migration 2026-09-10', NULL, false),
  ('saa_insight',        true,  true,  5,  'saa-team',  false,
     'dev output_type. deactivated for prod migration 2026-09-10 — replaced by saa-display', 144, false),
  ('sev2_insight',       true,  true,  1,  'saa-team',  false,
     'dev output_type. deactivated for prod migration 2026-09-10 — replaced by sev2-insights', 144, false),
  ('watchout_narratives',true,  true,  0,  'ish-team',  false,
     'dev output_type. deactivated for prod migration 2026-09-10 — single call 2026-08-12', NULL, false),
  -- prod capabilities (active)
  ('saa-display',        true,  true,  100, 'saa-team',  true,
     'prod output_type=saa-display. ~2,633 rows. is_groundable=false (hallucination detector dropped).', 2, false),
  ('situational-awareness', true, true, 100, 'saa-team', true,
     'prod output_type=situational-awareness. ~2,633 rows. is_groundable=false.', 2, false),
  ('sev2-insights',      true,  true,  10,  'saa-team',  true,
     'prod output_type=sev2-insights. ~572 rows. silence_grace_hours=NULL (irregular cadence, '
     'natural 100+ hour gaps).', NULL, false),
  ('summary',            true,  true,  1,   'ish-team',  true,
     'prod output_type=summary. saa->ish-eos scheduler. is_groundable=false (no prompts in prod).', 36, false)
AS t(capability, is_generative, is_gxp_relevant, expected_min_daily, owner, active, notes,
     silence_grace_hours, is_groundable)
LEFT ANTI JOIN mq_gmdf_dev.oil_obs.capability_registry existing
  ON existing.capability = t.capability;

-- Deactivate old dev capabilities that may already exist with active=true from prior seeding.
UPDATE mq_gmdf_dev.oil_obs.capability_registry
SET active = false,
    is_groundable = false,
    notes = concat(coalesce(notes, ''), ' | deactivated for prod migration 2026-09-10')
WHERE capability IN ('saa_insight', 'sev2_insight', 'watchout_narratives')
  AND active = true;

-- Update existing summary row to prod settings (is_groundable=false, prod notes).
UPDATE mq_gmdf_dev.oil_obs.capability_registry
SET is_groundable = false,
    notes = 'prod output_type=summary. saa->ish-eos scheduler. is_groundable=false (no prompts in prod).'
WHERE capability = 'summary'
  AND is_groundable = true;

-- Set the sev2-insights silence ceiling. Originally seeded 168h/7 days (item #2, 2026-09-21).
-- Tightened to 120h same day (FP/FN bias review follow-up, signed off): both 120h and 168h have
-- the identical 0/989 empirical historical false-positive rate (max gap ever observed 102.5h,
-- ~17.5h/17% margin at 120h), but 120h cuts the false-negative exposure window by ~29%.
-- Idempotent -- re-running just reasserts the same value.
UPDATE mq_gmdf_dev.oil_obs.capability_registry
SET silence_ceiling_hours = 120
WHERE capability = 'sev2-insights';

In [ ]:
%sql
-- SAFE to re-run: CREATE TABLE IF NOT EXISTS. This is the single table every detector's finding
-- ultimately lands in, and the only table ptof_obs_alert.ipynb's Teams-notify query reads from.
--
-- obs_incidents — append-only finding record.
-- MERGE keyed on (detector, source_row_id) so re-detecting the same finding updates rather than
-- duplicates. That gives dedup for free and makes acknowledgement stick across runs.
CREATE TABLE IF NOT EXISTS mq_gmdf_dev.oil_obs.obs_incidents (
    detector         STRING,        -- which detector produced this
    source_row_id    STRING,        -- the id being flagged; stable across runs
    capability       STRING,
    severity         STRING,        -- CRITICAL | WARN | INFO
    first_detected   TIMESTAMP,     -- set once, never updated
    last_detected    TIMESTAMP,     -- refreshed each time the finding is still present
    detection_count  BIGINT,        -- how many runs have seen it
    signal_payload   STRING,        -- JSON detail for triage
    notified_at      TIMESTAMP,     -- stamped when the alert reports it
    acknowledged_by  STRING,        -- set by a human, by hand
    acknowledged_at  TIMESTAMP,
    resolved_at      TIMESTAMP      -- set by hand once remediated
) CLUSTER BY (first_detected, detector);

In [ ]:
-- SAFE to re-run: CREATE TABLE IF NOT EXISTS + anti-join insert, guarded per check_name.
-- Seeds threshold_basis with documentation for every detection threshold in this system.
-- Retired detector thresholds are kept for historical reference with status = 'not_applicable_prod'.
-- New ETL pipeline thresholds added for the prod migration.
CREATE TABLE IF NOT EXISTS mq_gmdf_dev.oil_obs.threshold_basis (
    check_name STRING,
    threshold  STRING,
    basis      STRING,
    set_on     STRING,
    status     STRING
);

INSERT INTO mq_gmdf_dev.oil_obs.threshold_basis
SELECT t.* FROM VALUES
  -- active prod thresholds
  ('blank_output',
   '(>3 blank_count AND >=10 total_calls AND >2% blank_rate) OR (2-9 total_calls AND '
   '100% blank_count), 6h window',
   'Rate-path floor on blank_output_findings so an isolated blank response does not page. '
   'Low-volume bypass added 2026-09-21 (FP/FN bias review, priority 4, signed off): the rate '
   'path needs >=10 total_calls to evaluate at all, which structurally can never fire for a '
   'low-traffic capability. 2+ calls, 100% blank is a hard floor independent of the rate/volume '
   'floor -- a low-volume capability whose every recent call came back blank cannot be a fluke. '
   'Feeds a CRITICAL Teams alert.',
   '2026-09-09', 'unvalidated'),
  ('etl_pipeline_staleness', 'max(run_timestamp) < now() - INTERVAL 30 MINUTES',
   'The upstream ETL refreshes ~19 tables every 10-15 min. If no run completes for 30 min, '
   'the SAA agent is running on stale data. Feeds a CRITICAL check in the alert notebook.',
   '2026-09-10', 'provisional'),
  ('etl_table_staleness', 'minutes_since_last_run > 60, per table_or_view',
   'Per-table companion to etl_pipeline_staleness (which is a global max across all 19 '
   'tables). Added after the 2026-09-19/20 weekend incident, where 14 of 19 tables stalled '
   '~25h while the other 5 kept the global max fresh, masking the stall entirely. Runs '
   'alongside the global check, not replacing it -- see HANDOFF.md for the '
   'supplement-vs-replace reasoning. 60 min is ~4-5x the real ~13-16 min per-table cadence '
   'measured across all 19 tables, comfortably clear of the slowest single run ever observed '
   '(878s / ~14.6 min).',
   '2026-09-21', 'provisional'),
  ('etl_row_count_anomaly',
   'rows_written breaks the table_or_view''s all-time regime (expect_zero: pct_zero>=0.99, or '
   'expect_nonzero: pct_zero<=0.01, over full history, floor n>=50), trailing 24h window',
   'Closes a blind spot shared by etl_pipeline_failure/etl_table_staleness/etl_run_slow, none '
   'of which look at rows_written -- a run can report status=success on schedule in a normal '
   'amount of time while silently writing a partial or no-op load. Live-data confirmed '
   '2026-09-21: across ~45,000 historical runs the pattern is cleanly bimodal (14 source '
   'tables always write >0 rows, 5 materialized-view/gate tables always write exactly 0, zero '
   'crossovers ever) -- not an arbitrary row-count threshold, a first-ever-crossing detector. '
   'Shipped CRITICAL and digest-routed from day one (not per-incident first): its per-table, '
   'ETL-fleet-wide fan-out shape is analogous to etl_table_staleness''s real 14/19-table '
   'simultaneous event, so starting in digest mode avoids repeating that same '
   '"convert after the flood" lesson.',
   '2026-09-21', 'provisional'),
  ('capability_silence', 'hours_since_last_call > silence_grace_hours, per capability with '
   'silence_grace_hours IS NOT NULL',
   'Promoted WARN -> CRITICAL 2026-09-21 (detector value audit, evidence-based): full '
   'historical call-gap distribution queried per capability -- saa-display and '
   'situational-awareness (4,815 gaps each) never exceeded 0.27h against a 2h grace (7.4x '
   'margin); summary (96 gaps) never exceeded 12.04h against a 36h grace (3.0x margin). Zero '
   'empirical false positives across the entire history for any of the 3 covered '
   'capabilities. WARN-forever was hiding the exact single-capability-outage failure mode '
   'this detector exists to catch, against a threshold with real historical margin. Now '
   'table-backed via INCIDENT_SOURCES (finding_signature = sha2(capability, 256)) rather than '
   'a plain check().',
   '2026-09-21', 'provisional'),
  ('capability_silence_ceiling', 'hours_since_last_call > silence_ceiling_hours, per capability '
   'with silence_grace_hours IS NULL',
   'Backstop for capabilities structurally excluded from capability_silence because '
   'silence_grace_hours IS NULL (irregular cadence) -- confirmed gap (2026-09-21 audit): '
   'sev2-insights had gone silent 64h+ with no detection path at any length. Originally seeded '
   'WARN/168h (item #2, 2026-09-21). Promoted CRITICAL and tightened to silence_ceiling_hours=120 '
   'same day (FP/FN bias review follow-up, signed off): WARN-forever was itself judged a '
   'false-negative risk for a permanent-dark event. 120h and 168h have the identical 0/989 '
   'empirical historical false-positive rate (max gap ever observed 102.5h, ~17.5h/17% margin '
   'at 120h, on a thin sample of ~572 rows total), but 120h cuts the false-negative exposure '
   'window by ~29%. Has not yet fired on a real occurrence (unlike etl_run_slow before its '
   'promotion) -- revisit once it has live-fire history.',
   '2026-09-21', 'provisional'),
  ('etl_pipeline_failure', 'any failure in etl_pipeline_health (24h window)',
   'Any ETL task failure means at least one source table did not refresh. The agent''s outputs '
   'look normal but the underlying data is stale — capability_silence won''t fire.',
   '2026-09-10', 'provisional'),
  ('handover_delivery_new_failure',
   'any handover_delivery_failures row not yet recorded+acknowledged in obs_incidents',
   'Catches a single new occurrence — each one is a real shift handover that did not reach '
   'PFS3_ISH_SME@lists.lilly.com.',
   '2026-09-09', 'provisional'),
  ('handover_delivery_rate', '(failure_pct_7d > 20 AND attempts >= 10) OR failed >= 3',
   'Prod rate is 0% (0/49 all-time since 2026-08-27, 0/14 last 7d) against the 20% trip '
   'threshold -- large headroom by design, not evidence the check is miscalibrated. Superseded '
   '2026-09-21 (item #5) the pre-migration dev-era basis text (9.3% baseline / 12.5% current), '
   'which described dev data, not prod. This detector is a trend/escalation signal layered on '
   'top of handover_delivery (which already fires per-occurrence on every single failed send, '
   'keyed on ish_row_id) -- it exists to catch the underlying failure mechanism getting '
   'systemically worse, not to duplicate per-event detection. Absolute-count OR-trigger '
   '(failed >= 3) added 2026-09-21 (FP/FN bias review, priority 4, signed off): the rate path '
   'needs >=10 attempts to evaluate at all, which structurally can''t fire during a low-volume '
   'week (current pace ~14 attempts/7d) no matter how bad a small failure cluster is.',
   '2026-09-21', 'provisional'),
  ('long_running_incident', 'first_detected <= now() - INTERVAL 3 DAYS',
   'Three days unacknowledged is a real signal. Age-based, not detection_count-based. '
   'Promoted WARN -> CRITICAL 2026-09-21 (principle-based, not empirically validated -- '
   'obs_incidents had 0 rows at promotion time): fixes a pre-existing bug where this check was '
   'already labeled CRITICAL in code but never wired into '
   'INCIDENT_SOURCES/SCALAR_INCIDENT_SOURCES, so it never persisted or notified regardless of '
   'its severity label. Bounded-FP-by-construction: can only fire on top of an already-real, '
   'already-unresolved incident from another detector, so it introduces no new false-positive '
   'class of its own. check() moved into cell-4 (pre-persistence) so SCALAR_INCIDENT_SOURCES '
   'can pick up its result; consequently reads the previous run''s obs_incidents state rather '
   'than the current run''s -- an accepted ~5-10 min staleness against its 3-day age window.',
   '2026-09-21', 'revised'),
  ('nightly_baseline_staleness', 'computed_at < now() - INTERVAL 36 HOURS',
   'Nightly baseline job runs once/day; 36h gives ~12h grace past the nightly schedule. '
   'Promoted WARN -> CRITICAL 2026-09-21 (detector value audit, evidence-based): job run '
   'history shows the nightly baseline job completing reliably within schedule with no '
   'observed overrun approaching 36h -- WARN-only meant a stalled/broken nightly job (which '
   'silently degrades every threshold that reads from response_field_baseline / '
   'etl_duration_baseline) had no path to a human. check() moved into cell-4 (pre-persistence) '
   'so SCALAR_INCIDENT_SOURCES can pick up its result.',
   '2026-09-21', 'provisional'),
  ('pipeline_heartbeat', '0 rows in v_llm_bronze, trailing 45 minutes',
   'Catches a total upstream ingestion outage that every other detector implicitly assumes '
   'cannot happen. Tightened from 2h to 45min 2026-09-21 (FP/FN bias review, priority 5, '
   'signed off): real max gap in v_llm_bronze over 30 days is 16.4 min; 45 min is still 2.7x '
   'that (comfortable false-positive margin) but cuts the false-negative exposure window by '
   '~63% vs. the prior 2h.',
   '2026-09-09', 'provisional'),
  ('response_baseline_min_rows', '>= 20 eligible rows per capability',
   'Guard on response_field_baseline: a capability with fewer than 20 eligible rows is excluded '
   'from the baseline rather than inferring a schema shape from too few samples.',
   '2026-09-05', 'provisional'),
  ('schema_field_missing', 'current_present = 0 AND baseline_presence_rate >= 0.2',
   'A field that appeared in >= 20% of baseline rows disappearing entirely is drift. Rarely- '
   'populated optional fields are excluded from noise.',
   '2026-09-09', 'provisional'),
  ('unacknowledged_critical', 'severity = CRITICAL AND acknowledged_at IS NULL AND resolved_at IS NULL',
   'Backstop rollup across every detector: catches anything already flagged that nobody has '
   'acted on. Promoted WARN -> CRITICAL 2026-09-21 (principle-based, not empirically '
   'validated -- obs_incidents had 0 rows at promotion time): fixes a pre-existing bug where '
   'this check was already labeled CRITICAL in code but never wired into '
   'INCIDENT_SOURCES/SCALAR_INCIDENT_SOURCES, so it never persisted or notified regardless of '
   'its severity label. Bounded-FP-by-construction: can only fire on top of an already-real '
   'CRITICAL incident from another detector, so it introduces no new false-positive class of '
   'its own. check() moved into cell-4 (pre-persistence) so SCALAR_INCIDENT_SOURCES can pick '
   'up its result; consequently reads the previous run''s obs_incidents state rather than the '
   'current run''s -- an accepted ~5-10 min staleness, immaterial against its unbounded-age '
   'window.',
   '2026-09-21', 'provisional'),
  ('write_lag_anomalies',
   'write_lag_s > greatest(median + 5*1.4826*MAD (per capability x scheduler_run, floor n>=50, '
   '30d window), 300), rolled up to >=3 anomalous rows in a 60-min bin',
   'MAD instead of mean/stddev because write_lag_s is right-skewed. Distinct from '
   'capability_silence (stopped writing) -- this was meant to catch "still writing, slower '
   'than usual." 300s floor added 2026-09-21 (item #3, signed off): the MAD baseline is '
   'degenerate (median=MAD=0 for every capability x scheduler_run, since write_lag_s is 0 for '
   'almost all prod rows), so the raw threshold was effectively "write_lag_s > 0" -- the floor '
   'ensured only genuinely extreme lag (5+ min) could trip this. Not a substitute for true '
   'per-call inference latency (latency_ms), which is blocked on '
   'ptof_primary__ai_llm_audit_log having 0 rows in prod. RETIRED 2026-09-21 (detector value '
   'audit): write_lag_s confirmed exactly 0 for all 10,721 rows over the full 30-day window -- '
   'zero variance, not a threshold problem. Whatever the floor, the signal carried no '
   'information in prod, whether that reflects a genuinely-instant write path or an unwired '
   'ingestion_ts column upstream. Table and check() removed entirely from the codebase rather '
   'than left WARN-forever with nothing to promote -- a fresh detector can be reintroduced if '
   'write-latency instrumentation is ever wired up for real.',
   '2026-09-18', 'not_applicable_prod'),
  ('etl_run_slow',
   'duration_seconds > median + 5*1.4826*MAD (per table_or_view x task_name, status=success '
   'only, floor n>=50, 30d window), rolled up to >=3 slow runs in a 60-min bin, grouped by '
   '(task_name, window_start)',
   'Same MAD rationale as write_lag_anomalies. Distinct from etl_pipeline_failure '
   '(outright failure) and etl_pipeline_staleness (no run at all) -- this is "runs '
   'completing, but slower than usual," an early warning that can precede either of '
   'those. Grain changed 2026-09-21 (item #4, signed off) from '
   '(table_or_view, task_name, window_start) to (task_name, window_start), with '
   'affected_tables/affected_table_count in the payload: audit found all tables under '
   'one task_name always move together (4 real events in 7 days each hit 14-19 of 19 '
   'tables simultaneously), so the old per-table grain produced 14-19 near-duplicate '
   'rows per real event. Promoted WARN -> CRITICAL 2026-09-21 (FP/FN bias '
   'review, priority 2, signed off): 5 real, correlated, multi-table slowdown events observed '
   'in ~1 week while WARN/log-only -- a proven-real signal that was invisible to a '
   'human. Now persists to obs_incidents and posts to Teams like the other CRITICAL '
   'detectors.',
   '2026-09-18', 'provisional'),
  -- retired detector thresholds (kept for historical reference)
  ('capability_error_rate', 'error_rate = 1.0 & n>=5, or > 0.20 & n>=10',
   'observed 0.00 or 0.93-1.00; no middle ground in data', '2026-08-20', 'not_applicable_prod'),
  ('capability_error_rate_sustained', '0.50',
   'Bimodal failure distribution (0% or 93%+). Dropped: no success/error_msg in prod.',
   '2026-09-03', 'not_applicable_prod'),
  ('capability_silence dsa_copilot', 'silence_grace_hours = 26',
   'Dev-only capability, deactivated.', '2026-08-20', 'not_applicable_prod'),
  ('capability_silence dsa_optimize', 'silence_grace_hours = 2',
   'Dev-only capability, deactivated.', '2026-08-20', 'not_applicable_prod'),
  ('credential_outage', 'sum(fastfail_calls) > 0, current day',
   'Dev-specific model (demo-claude-sonnet-4-6-pwc-omi) does not exist in prod.',
   '2026-09-09', 'not_applicable_prod'),
  ('hallucination medium', 'pctile < 0.05 AND similarity < 0.75',
   'Dropped: 4.9% false positive rate on computed numbers.', '2026-08-20', 'not_applicable_prod'),
  ('hallucination_similarity_cross_check', 'resp_vs_prompt_similarity < 0.80',
   'Dropped with hallucination detector.', '2026-09-05', 'not_applicable_prod'),
  ('hallucination_unverified_rate', 'count>=10 AND unverified_rate > 0.5, 24h window',
   'Dropped with hallucination detector.', '2026-09-09', 'not_applicable_prod'),
  ('latency_anomaly', 'p95 + 3*IQR, is_reliable only',
   'Dropped: no latency_ms in prod. Phase 2 via dev enrichment.', '2026-08-20', 'not_applicable_prod'),
  ('latency_anomaly_findings_min_count', '>= 2 anomalous calls in the detection window',
   'Dropped with latency_anomaly.', '2026-09-04', 'not_applicable_prod'),
  ('latency_anomaly_unreliable_baseline', 'same anomaly verdicts, is_reliable = false',
   'Dropped with latency_anomaly.', '2026-09-09', 'not_applicable_prod'),
  ('latency_baseline_missing', 'successful calls exist with no capability_latency_baseline row',
   'Dropped: no latency_ms in prod.', '2026-09-09', 'not_applicable_prod'),
  ('latency_baseline_n_samples', '>= 30',
   'Dropped with latency_anomaly.', '2026-09-05', 'not_applicable_prod'),
  ('latency_baseline_span_days', '>= 6 distinct calendar days with a call',
   'Dropped with latency_anomaly.', '2026-09-05', 'not_applicable_prod'),
  ('latency_fixed_ceiling', 'latency_ms > 30000, successful calls',
   'Dropped: no latency_ms in prod.', '2026-09-09', 'not_applicable_prod'),
  ('prompt_size_drift_multiplier', '2x nightly baseline p95 (prompt or response chars)',
   'Dropped: no user_prompt/system_prompt in prod.', '2026-09-04', 'not_applicable_prod'),
  ('rapid_human_correction',
   'AI publish and ISH correction match within 10 minutes',
   'Dropped: 6 rows total, dormant, research signal.', '2026-09-01', 'not_applicable_prod'),
  ('runtime_allowlist_populated', '0 rows in runtime_allowlist for the running environment',
   'Dropped: single transport (cortex), zero violations.', '2026-09-09', 'not_applicable_prod'),
  ('runtime_observed_digest_floor', 'occurrences_7d >= 5',
   'Dropped with weekly runtime digest.', '2026-08-27', 'not_applicable_prod'),
  ('runtime_violation_digest_cadence', 'weekly, Monday 08:00 America/Indianapolis',
   'Dropped with weekly runtime digest.', '2026-08-27', 'not_applicable_prod'),
  ('runtime_violation_immediate', 'transport OR model_config sanctioned for no capability in env',
   'Dropped: single transport.', '2026-08-27', 'not_applicable_prod'),
  ('ungrounded_token_count', '> 3 AND is_gxp_relevant',
   'Dropped with hallucination detector.', '2026-08-20', 'not_applicable_prod'),
  ('write_lag', 'p95_ingest_only_s > 10',
   'Dropped: write_lag_daily depends on latency_ms for ingest-only computation.', '2026-08-20', 'not_applicable_prod')
AS t(check_name, threshold, basis, set_on, status)
LEFT ANTI JOIN mq_gmdf_dev.oil_obs.threshold_basis existing
  ON existing.check_name = t.check_name;

-- Item #5 (signed off 2026-09-21): the anti-join INSERT above only seeds a row if check_name
-- doesn't already exist -- it will NOT overwrite handover_delivery_rate's basis text in an
-- environment where threshold_basis was already seeded with the old pre-migration dev numbers
-- (9.3%/12.5%). This UPDATE corrects that existing row explicitly; safe to re-run (idempotent).
UPDATE mq_gmdf_dev.oil_obs.threshold_basis
SET threshold = '(failure_pct_7d > 20 AND attempts >= 10) OR failed >= 3',
    basis = 'Prod rate is 0% (0/49 all-time since 2026-08-27, 0/14 last 7d) against the 20% '
            'trip threshold -- large headroom by design, not evidence the check is '
            'miscalibrated. Superseded 2026-09-21 (item #5) the pre-migration dev-era basis '
            'text (9.3% baseline / 12.5% current), which described dev data, not prod. This '
            'detector is a trend/escalation signal layered on top of handover_delivery (which '
            'already fires per-occurrence on every single failed send, keyed on ish_row_id) -- '
            'it exists to catch the underlying failure mechanism getting systemically worse, '
            'not to duplicate per-event detection. Absolute-count OR-trigger (failed >= 3) '
            'added 2026-09-21 (FP/FN bias review, priority 4, signed off): the rate path needs '
            '>=10 attempts to evaluate at all, which structurally can''t fire during a '
            'low-volume week (current pace ~14 attempts/7d) no matter how bad a small failure '
            'cluster is.',
    set_on = '2026-09-21',
    status = 'provisional'
WHERE check_name = 'handover_delivery_rate';

-- FP/FN bias review follow-ups (2026-09-21, signed off) -- correct rows that may already exist
-- in an environment seeded before this session, same idempotent-UPDATE pattern as item #5 above.
UPDATE mq_gmdf_dev.oil_obs.threshold_basis
SET threshold = '0 rows in v_llm_bronze, trailing 45 minutes',
    basis = 'Catches a total upstream ingestion outage that every other detector implicitly '
            'assumes cannot happen. Tightened from 2h to 45min 2026-09-21 (FP/FN bias review, '
            'priority 5, signed off): real max gap in v_llm_bronze over 30 days is 16.4 min; '
            '45 min is still 2.7x that (comfortable false-positive margin) but cuts the '
            'false-negative exposure window by ~63% vs. the prior 2h.',
    set_on = '2026-09-21'
WHERE check_name = 'pipeline_heartbeat';

UPDATE mq_gmdf_dev.oil_obs.threshold_basis
SET threshold = '(>3 blank_count AND >=10 total_calls AND >2% blank_rate) OR (2-9 total_calls '
                 'AND 100% blank_count), 6h window',
    basis = 'Rate-path floor on blank_output_findings so an isolated blank response does not '
            'page. Low-volume bypass added 2026-09-21 (FP/FN bias review, priority 4, signed '
            'off): the rate path needs >=10 total_calls to evaluate at all, which structurally '
            'can never fire for a low-traffic capability. 2+ calls, 100% blank is a hard floor '
            'independent of the rate/volume floor -- a low-volume capability whose every '
            'recent call came back blank cannot be a fluke. Feeds a CRITICAL Teams alert.',
    set_on = '2026-09-21'
WHERE check_name = 'blank_output';

UPDATE mq_gmdf_dev.oil_obs.threshold_basis
SET threshold = 'duration_seconds > median + 5*1.4826*MAD (per table_or_view x task_name, '
                 'status=success only, floor n>=50, 30d window), rolled up to >=3 slow runs '
                 'in a 60-min bin, grouped by (task_name, window_start)',
    basis = 'Same MAD rationale as write_lag_anomalies. Distinct from etl_pipeline_failure '
            '(outright failure) and etl_pipeline_staleness (no run at all) -- this is "runs '
            'completing, but slower than usual," an early warning that can precede either of '
            'those. Grain changed 2026-09-21 (item #4, signed off) from '
            '(table_or_view, task_name, window_start) to (task_name, window_start), with '
            'affected_tables/affected_table_count in the payload: audit found all tables under '
            'one task_name always move together (4 real events in 7 days each hit 14-19 of 19 '
            'tables simultaneously), so the old per-table grain produced 14-19 near-duplicate '
            'rows per real event. Promoted WARN -> CRITICAL 2026-09-21 (FP/FN bias review, '
            'priority 2, signed off): 5 real, correlated, multi-table slowdown events observed '
            'in ~1 week while WARN/log-only -- a proven-real signal that was invisible to a '
            'human. Now persists to obs_incidents and posts to Teams like the other CRITICAL '
            'detectors.',
    status = 'provisional',
    set_on = '2026-09-21'
WHERE check_name = 'etl_run_slow';

-- capability_silence_ceiling promotion (2026-09-21, same day as item #2, signed off) -- correct
-- the row in an environment already seeded with the original WARN/168h text, same idempotent
-- pattern as the UPDATEs above.
UPDATE mq_gmdf_dev.oil_obs.threshold_basis
SET threshold = 'hours_since_last_call > silence_ceiling_hours, per capability with '
                'silence_grace_hours IS NULL',
    basis = 'Backstop for capabilities structurally excluded from capability_silence because '
            'silence_grace_hours IS NULL (irregular cadence) -- confirmed gap (2026-09-21 '
            'audit): sev2-insights had gone silent 64h+ with no detection path at any length. '
            'Originally seeded WARN/168h (item #2, 2026-09-21). Promoted CRITICAL and '
            'tightened to silence_ceiling_hours=120 same day (FP/FN bias review follow-up, '
            'signed off): WARN-forever was itself judged a false-negative risk for a '
            'permanent-dark event. 120h and 168h have the identical 0/989 empirical historical '
            'false-positive rate (max gap ever observed 102.5h, ~17.5h/17% margin at 120h, on '
            'a thin sample of ~572 rows total), but 120h cuts the false-negative exposure '
            'window by ~29%. Has not yet fired on a real occurrence (unlike etl_run_slow '
            'before its promotion) -- revisit once it has live-fire history.',
    set_on = '2026-09-21',
    status = 'provisional'
WHERE check_name = 'capability_silence_ceiling';

-- Detector value audit (2026-09-21, signed off) -- correct the 4 remaining rows whose promotion/
-- retirement text and status the anti-join INSERT above cannot update in an environment where
-- threshold_basis was seeded before this audit. Same idempotent pattern as the UPDATEs above.
UPDATE mq_gmdf_dev.oil_obs.threshold_basis
SET basis = 'Nightly baseline job runs once/day; 36h gives ~12h grace past the nightly '
            'schedule. Promoted WARN -> CRITICAL 2026-09-21 (detector value audit, '
            'evidence-based): job run history shows the nightly baseline job completing '
            'reliably within schedule with no observed overrun approaching 36h -- WARN-only '
            'meant a stalled/broken nightly job (which silently degrades every threshold that '
            'reads from response_field_baseline / etl_duration_baseline) had no path to a '
            'human. check() moved into cell-4 (pre-persistence) so SCALAR_INCIDENT_SOURCES can '
            'pick up its result.',
    set_on = '2026-09-21'
WHERE check_name = 'nightly_baseline_staleness';

UPDATE mq_gmdf_dev.oil_obs.threshold_basis
SET basis = 'Backstop rollup across every detector: catches anything already flagged that '
            'nobody has acted on. Promoted WARN -> CRITICAL 2026-09-21 (principle-based, not '
            'empirically validated -- obs_incidents had 0 rows at promotion time): fixes a '
            'pre-existing bug where this check was already labeled CRITICAL in code but never '
            'wired into INCIDENT_SOURCES/SCALAR_INCIDENT_SOURCES, so it never persisted or '
            'notified regardless of its severity label. Bounded-FP-by-construction: can only '
            'fire on top of an already-real CRITICAL incident from another detector, so it '
            'introduces no new false-positive class of its own. check() moved into cell-4 '
            '(pre-persistence) so SCALAR_INCIDENT_SOURCES can pick up its result; consequently '
            'reads the previous run''s obs_incidents state rather than the current run''s -- '
            'an accepted ~5-10 min staleness, immaterial against its unbounded-age window.',
    set_on = '2026-09-21'
WHERE check_name = 'unacknowledged_critical';

UPDATE mq_gmdf_dev.oil_obs.threshold_basis
SET basis = 'Three days unacknowledged is a real signal. Age-based, not detection_count-based. '
            'Promoted WARN -> CRITICAL 2026-09-21 (principle-based, not empirically '
            'validated -- obs_incidents had 0 rows at promotion time): fixes a pre-existing '
            'bug where this check was already labeled CRITICAL in code but never wired into '
            'INCIDENT_SOURCES/SCALAR_INCIDENT_SOURCES, so it never persisted or notified '
            'regardless of its severity label. Bounded-FP-by-construction: can only fire on '
            'top of an already-real, already-unresolved incident from another detector, so it '
            'introduces no new false-positive class of its own. check() moved into cell-4 '
            '(pre-persistence) so SCALAR_INCIDENT_SOURCES can pick up its result; consequently '
            'reads the previous run''s obs_incidents state rather than the current run''s -- '
            'an accepted ~5-10 min staleness against its 3-day age window.',
    set_on = '2026-09-21'
WHERE check_name = 'long_running_incident';

UPDATE mq_gmdf_dev.oil_obs.threshold_basis
SET basis = 'MAD instead of mean/stddev because write_lag_s is right-skewed. Distinct from '
            'capability_silence (stopped writing) -- this was meant to catch "still writing, '
            'slower than usual." 300s floor added 2026-09-21 (item #3, signed off): the MAD '
            'baseline is degenerate (median=MAD=0 for every capability x scheduler_run, since '
            'write_lag_s is 0 for almost all prod rows), so the raw threshold was effectively '
            '"write_lag_s > 0" -- the floor ensured only genuinely extreme lag (5+ min) could '
            'trip this. Not a substitute for true per-call inference latency (latency_ms), '
            'which is blocked on ptof_primary__ai_llm_audit_log having 0 rows in prod. '
            'RETIRED 2026-09-21 (detector value audit): write_lag_s confirmed exactly 0 for '
            'all 10,721 rows over the full 30-day window -- zero variance, not a threshold '
            'problem. Whatever the floor, the signal carried no information in prod, whether '
            'that reflects a genuinely-instant write path or an unwired ingestion_ts column '
            'upstream. Table and check() removed entirely from the codebase rather than left '
            'WARN-forever with nothing to promote -- a fresh detector can be reintroduced if '
            'write-latency instrumentation is ever wired up for real.',
    set_on = '2026-09-21',
    status = 'not_applicable_prod'
WHERE check_name = 'write_lag_anomalies';

In [ ]:
%sql
-- ONE-TIME cleanup, safe to re-run (all DROPs are IF EXISTS). Removes tables and views orphaned
-- by the prod migration: detector output tables from dropped detectors, reference tables that are
-- no longer read, and older v1.1 orphans. Order: views first, then tables.

-- views
DROP VIEW IF EXISTS mq_gmdf_dev.oil_obs.v_ungrounded_tokens;

-- detector output tables from dropped detectors
DROP TABLE IF EXISTS mq_gmdf_dev.oil_obs.hallucination_signal;
DROP TABLE IF EXISTS mq_gmdf_dev.oil_obs.faithfulness_scores;
DROP TABLE IF EXISTS mq_gmdf_dev.oil_obs.latency_anomalies;
DROP TABLE IF EXISTS mq_gmdf_dev.oil_obs.latency_anomaly_findings;
DROP TABLE IF EXISTS mq_gmdf_dev.oil_obs.capability_error_rate_alert;
DROP TABLE IF EXISTS mq_gmdf_dev.oil_obs.capability_error_rate_findings;
DROP TABLE IF EXISTS mq_gmdf_dev.oil_obs.capability_health;
DROP TABLE IF EXISTS mq_gmdf_dev.oil_obs.credential_fastfail_daily;
DROP TABLE IF EXISTS mq_gmdf_dev.oil_obs.latency_failures;
DROP TABLE IF EXISTS mq_gmdf_dev.oil_obs.prompt_size_drift;
DROP TABLE IF EXISTS mq_gmdf_dev.oil_obs.write_lag_daily;
DROP TABLE IF EXISTS mq_gmdf_dev.oil_obs.transport_violation_signatures;
DROP TABLE IF EXISTS mq_gmdf_dev.oil_obs.ish_entity_dim;
DROP TABLE IF EXISTS mq_gmdf_dev.oil_obs.rapid_human_correction;

-- detector output tables retired 2026-09-21 (dead-code audit) -- their headers in
-- ptof_obs_liveness_detection.ipynb/ptof_obs_nightly_baseline.ipynb/
-- ptof_obs_behavioral_correlation.ipynb had already documented these as removed, but no DROP
-- statement existed here yet to actually clean up the underlying table. Added now to close
-- that gap.
DROP TABLE IF EXISTS mq_gmdf_dev.oil_obs.write_lag_anomalies;
DROP TABLE IF EXISTS mq_gmdf_dev.oil_obs.write_lag_baseline;
DROP TABLE IF EXISTS mq_gmdf_dev.oil_obs.handover_delivery_rate_findings;

-- reference tables no longer read by any active detector
DROP TABLE IF EXISTS mq_gmdf_dev.oil_obs.runtime_observed;
DROP TABLE IF EXISTS mq_gmdf_dev.oil_obs.runtime_allowlist;
DROP TABLE IF EXISTS mq_gmdf_dev.oil_obs._obs_watermark;

-- older v1.1 orphans (kept here for completeness)
DROP TABLE IF EXISTS mq_gmdf_dev.oil_obs.hallucination_verdicts;
DROP TABLE IF EXISTS mq_gmdf_dev.oil_obs.blank_output_incidents;
DROP TABLE IF EXISTS mq_gmdf_dev.oil_obs.transport_violations;
DROP TABLE IF EXISTS mq_gmdf_dev.oil_obs.capability_outage_findings;
DROP TABLE IF EXISTS mq_gmdf_dev.oil_obs.response_schema_baseline;
DROP TABLE IF EXISTS mq_gmdf_dev.oil_obs.transport_allowlist;
DROP TABLE IF EXISTS mq_gmdf_dev.oil_obs.success_rate_daily;